In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [69.9 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14 http://ar

In [ ]:
spark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Hexaware").getOrCreate()
# driver program starts
#spark context created
#cluster connection established

In [ ]:
# Read text file
data = spark.read.text("file1.txt")


In [ ]:
data

DataFrame[value: string]

In [ ]:
print(data)

DataFrame[value: string]


In [ ]:
data.show(3)

+-----------------+
|            value|
+-----------------+
|    spark is fast|
|spark is powerful|
|rdd is foundation|
+-----------------+



In [ ]:
word count program
spark 2
is 3
fast 1
powerful 1
rdd 1
foundation 1

In [ ]:
import pandas as pd
df = pd.read_csv("file1.txt",header=None,names=["value"])
df.head()

,value
0,spark is fast
1,spark is powerful
2,rdd is foundation


In [ ]:
word_count = (df["value"].str.lower().str.split().explode().value_counts())
print(word_count)

value
is            3
spark         2
fast          1
powerful      1
rdd           1
foundation    1
Name: count, dtype: int64


In [ ]:
df["value"].str.lower().str.split().explode().value_counts()

,count
value,
is,3
spark,2
fast,1
powerful,1
rdd,1
foundation,1


In [ ]:
from pyspark.sql.functions import explode, split, col, lower
#step 1: conert sentences into words
words = data.select(explode(split(lower(col("value")), " ")).alias("word"))
words.show()

+----------+
|      word|
+----------+
|     spark|
|        is|
|      fast|
|     spark|
|        is|
|  powerful|
|       rdd|
|        is|
|foundation|
+----------+



In [ ]:
#step 2: count frequency
word_count = words.groupBy("word").count()
word_count.show()

+----------+-----+
|      word|count|
+----------+-----+
|  powerful|    1|
|      fast|    1|
|       rdd|    1|
|        is|    3|
|     spark|    2|
|foundation|    1|
+----------+-----+



In [ ]:
word_count = word_count.orderBy(col("count").desc())
word_count.show()

+----------+-----+
|      word|count|
+----------+-----+
|        is|    3|
|     spark|    2|
|  powerful|    1|
|      fast|    1|
|       rdd|    1|
|foundation|    1|
+----------+-----+



In [ ]:
data.printSchema()

root
 |-- value: string (nullable = true)



In [ ]:
lines = data.rdd.map(lambda row: row[0])

In [ ]:
lines.collect()

['data engineering is fun',
 'big data requires distributed processing',
 'hadoop mapreduce processes big data',
 'data engineering uses hadoop']

In [ ]:
words = lines.flatMap(lambda line: line.split(" "))

In [ ]:
words.collect()

['data',
 'engineering',
 'is',
 'fun',
 'big',
 'data',
 'requires',
 'distributed',
 'processing',
 'hadoop',
 'mapreduce',
 'processes',
 'big',
 'data',
 'data',
 'engineering',
 'uses',
 'hadoop']

In [ ]:
word_pairs = words.map(lambda word: (word, 1))
word_pairs.collect()

[('data', 1),
 ('engineering', 1),
 ('is', 1),
 ('fun', 1),
 ('big', 1),
 ('data', 1),
 ('requires', 1),
 ('distributed', 1),
 ('processing', 1),
 ('hadoop', 1),
 ('mapreduce', 1),
 ('processes', 1),
 ('big', 1),
 ('data', 1),
 ('data', 1),
 ('engineering', 1),
 ('uses', 1),
 ('hadoop', 1)]

In [ ]:
numbers=spark.sparkContext.parallelize([1,2,3,4,5])
result = numbers.reduce(lambda a, b: a + b)
print(result)

15


In [ ]:
word_counts = word_pairs.reduce(lambda a, b: a + b)
word_counts.collect()

AttributeError: 'tuple' object has no attribute 'collect'

In [ ]:


# Read text file
data = spark.read.text("file1.txt")

# Convert DataFrame to RDD
lines = data.rdd.map(lambda row: row[0])

# Split words
words = lines.flatMap(lambda line: line.split(" "))

# Map words to (word,1)
word_pairs = words.map(lambda word: (word, 1))

# Reduce by key to count words
word_counts = word_pairs.reduceByKey(lambda a, b: a + b)

# Collect results
results = word_counts.collect()

# Print results
for word, count in results:
    print(word, count)

# Stop Spark
spark.stop()

data 4
engineering 2
is 1
fun 1
big 2
requires 1
distributed 1
processing 1
hadoop 2
mapreduce 1
processes 1
uses 1


In [ ]:
rdd =sc.parallelize([1,2,3,4,5])

NameError: name 'sc' is not defined

In [ ]:
#In pyspark, every operation falls into two categories:
Transformations: used to transform data,
do not execute immediately
they are lazy(execution will be later)
i am defining what to do

# Actions
used to get results
trigger execution
now actually do it

In [ ]:
data = [("Alice",25), ("bob",30),("cathy",29)]
columns= ["Name","Age"]
df = spark.createDataFrame(data,columns)
df.show()

+-----+---+
| Name|Age|
+-----+---+
|Alice| 25|
|  bob| 30|
|cathy| 29|
+-----+---+



In [ ]:
# Transformation examples
select, filter, withcolumn
#action examples
show,count,collect

In [ ]:
df2=df.select("Name")

In [ ]:
from pyspark.sql.functions import col
#df3 = df.filter(col("Age")>25)
df3=df.filter(df.Age>25)
df4=df.withColumn("Age_plus_1",col("Age")+1)

In [ ]:
df2.show()

+-----+
| Name|
+-----+
|Alice|
|  bob|
|cathy|
+-----+



df.filter(df.Age>25).select("Name").show()
1. Build plan
2. optimiztion plan
3. execute at .show()

#common Transformation vs Actions
select
filter
withcolumn
groupby
join

#actions
show
count
collect
take
first